In [6]:
import os
import pandas as pd

print("Current folder:", os.getcwd())
print("Files here:", os.listdir("."))

Current folder: c:\Users\ACER\OneDrive\Desktop\radiology_project
Files here: ['dataset_full', 'dataset_sample', 'main.py', 'radiology.ipynb']


In [2]:
print("dataset_full contents:", os.listdir("dataset_full"))
print("Training contents:", os.listdir("dataset_full/Training"))

dataset_full contents: ['stage2_test_metadata.csv', 'stage2_train_metadata.csv', 'Test', 'Training']
Training contents: ['Images', 'Masks']


In [7]:
train_df = pd.read_csv("dataset_full/stage2_train_metadata.csv")

print("Train shape:", train_df.shape)
print("\nColumns:")
print(train_df.columns)


print("\nFirst 5 rows:")
print(train_df.head())

Train shape: (30227, 11)

Columns:
Index(['patientId', 'x', 'y', 'width', 'height', 'Target', 'class', 'age',
       'sex', 'modality', 'position'],
      dtype='object')

First 5 rows:
                              patientId      x      y  width  height  Target  \
0  0004cfab-14fd-4e49-80ba-63a80b6bddd6    NaN    NaN    NaN     NaN       0   
1  00313ee0-9eaa-42f4-b0ab-c148ed3241cd    NaN    NaN    NaN     NaN       0   
2  00322d4d-1c29-4943-afc9-b6754be640eb    NaN    NaN    NaN     NaN       0   
3  003d8fa0-6bf1-40ed-b54c-ac657f8495c5    NaN    NaN    NaN     NaN       0   
4  00436515-870c-4b36-a041-de91049b9ab4  264.0  152.0  213.0   379.0       1   

                          class  age sex modality position  
0  No Lung Opacity / Not Normal   51   F       CR       PA  
1  No Lung Opacity / Not Normal   48   F       CR       PA  
2  No Lung Opacity / Not Normal   19   M       CR       AP  
3                        Normal   28   M       CR       PA  
4                  Lung Opac

In [8]:
print(train_df["Target"].value_counts())

Target
0    20672
1     9555
Name: count, dtype: int64


In [9]:
train_df = pd.read_csv("dataset_full/stage2_train_metadata.csv")

print("Train shape:", train_df.shape)
print("\nColumns:")
print(train_df.columns)

print("\nFirst 5 rows:")
print(train_df.head())

Train shape: (30227, 11)

Columns:
Index(['patientId', 'x', 'y', 'width', 'height', 'Target', 'class', 'age',
       'sex', 'modality', 'position'],
      dtype='object')

First 5 rows:
                              patientId      x      y  width  height  Target  \
0  0004cfab-14fd-4e49-80ba-63a80b6bddd6    NaN    NaN    NaN     NaN       0   
1  00313ee0-9eaa-42f4-b0ab-c148ed3241cd    NaN    NaN    NaN     NaN       0   
2  00322d4d-1c29-4943-afc9-b6754be640eb    NaN    NaN    NaN     NaN       0   
3  003d8fa0-6bf1-40ed-b54c-ac657f8495c5    NaN    NaN    NaN     NaN       0   
4  00436515-870c-4b36-a041-de91049b9ab4  264.0  152.0  213.0   379.0       1   

                          class  age sex modality position  
0  No Lung Opacity / Not Normal   51   F       CR       PA  
1  No Lung Opacity / Not Normal   48   F       CR       PA  
2  No Lung Opacity / Not Normal   19   M       CR       AP  
3                        Normal   28   M       CR       PA  
4                  Lung Opac

In [10]:
image_labels = train_df.groupby("patientId")["Target"].max().reset_index()

print("Image-level shape:", image_labels.shape)
print(image_labels.head())
print("\nTarget counts:")
print(image_labels["Target"].value_counts())

Image-level shape: (26684, 2)
                              patientId  Target
0  0004cfab-14fd-4e49-80ba-63a80b6bddd6       0
1  000924cf-0f8d-42bd-9158-1af53881a557       0
2  000db696-cf54-4385-b10b-6b16fbb3f985       1
3  000fe35a-2649-43d4-b027-e67796d412e0       1
4  001031d9-f904-4a23-b3e5-2c088acd19c6       1

Target counts:
Target
0    20672
1     6012
Name: count, dtype: int64


In [11]:
normal_df = image_labels[image_labels["Target"] == 0].sample(500, random_state=42)
pneumonia_df = image_labels[image_labels["Target"] == 1].sample(500, random_state=42)

sample_df = pd.concat([normal_df, pneumonia_df]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Sample shape:", sample_df.shape)
print(sample_df["Target"].value_counts())
print(sample_df.head())

Sample shape: (1000, 2)
Target
1    500
0    500
Name: count, dtype: int64
                              patientId  Target
0  6e73017c-4e3a-499b-a597-4a797bb179de       1
1  d84d271a-8ea1-4065-90e5-9f9c94794310       1
2  3b2cd577-6e31-4b41-8349-dacac273aadf       1
3  04e9f692-f3d6-496b-ae0c-905137cc1f84       1
4  86b71e6a-ccf0-4ff8-bbeb-221a49e57f6d       0


In [12]:
import shutil
import os

source_folder = "datset_full/Training/Images"
target_folder = "dataset_sample/images"

os.makedirs(target_folder, exist_ok=True)

copied = 0

for _, row in sample_df.iterrows():
    img_name = row["patientId"] + ".png"
    src = os.path.join(source_folder, img_name)
    dst = os.path.join(target_folder, img_name)

    if os.path.exists(src):
        shutil.copy(src, dst)
        copied += 1

os.makedirs("dataset_sample", exist_ok=True)
sample_df.to_csv("dataset_sample/sample_labels.csv", index=False)

print("Copied images:", copied)
print("Saved labels file.")

Copied images: 0
Saved labels file.


In [13]:
import cv2
import numpy as np

df = pd.read_csv("dataset_sample/sample_labels.csv")

images = []
labels = []

for _, row in df.iterrows():
    img_name = row["patientId"] + ".png"
    img_path = os.path.join("dataset_sample/images", img_name)

    img = cv2.imread(img_path)
    if img is not None:
        img = cv2.resize(img, (224, 224))
        img = img / 255.0

        images.append(img)
        labels.append(row["Target"])

X = np.array(images)
y = np.array(labels)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1000, 224, 224, 3)
y shape: (1000,)


In [14]:
import os

print("Sample folder exists:", os.path.exists("dataset_sample/images"))
print("Number of copied images:", len(os.listdir("dataset_sample/images")) if os.path.exists("dataset_sample/images") else 0)

if os.path.exists("dataset_sample/images"):
    print("First 5 copied images:", os.listdir("dataset_sample/images")[:5])

Sample folder exists: True
Number of copied images: 1000
First 5 copied images: ['0100515c-5204-4f31-98e0-f35e4b00004a.png', '0101174b-6643-4d4e-b4ba-b6d41d0ce46a.png', '012a5620-d082-4bb8-9b3b-e72d8938000c.png', '01a7353d-25bb-4ff8-916b-f50dd541dccf.png', '01f3abc2-33c7-4ea2-a599-dc49b76fcfae.png']


In [15]:
import pandas as pd

df = pd.read_csv("dataset_sample/sample_labels.csv")
print("Sample labels shape:", df.shape)
print(df.head())


Sample labels shape: (1000, 2)
                              patientId  Target
0  6e73017c-4e3a-499b-a597-4a797bb179de       1
1  d84d271a-8ea1-4065-90e5-9f9c94794310       1
2  3b2cd577-6e31-4b41-8349-dacac273aadf       1
3  04e9f692-f3d6-496b-ae0c-905137cc1f84       1
4  86b71e6a-ccf0-4ff8-bbeb-221a49e57f6d       0


In [16]:
import pandas as pd
import os
import shutil

# Correct path
train_df = pd.read_csv("dataset_full/stage2_train_metadata.csv")

# One image = one label
image_labels = train_df.groupby("patientId")["Target"].max().reset_index()

# Balanced sample
normal_df = image_labels[image_labels["Target"] == 0].sample(500, random_state=42)
pneumonia_df = image_labels[image_labels["Target"] == 1].sample(500, random_state=42)

sample_df = pd.concat([normal_df, pneumonia_df]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Sample shape:", sample_df.shape)
print(sample_df["Target"].value_counts())

# Correct source folder
source_folder = "dataset_full/Training/Images"
target_folder = "dataset_sample/images"

os.makedirs(target_folder, exist_ok=True)

copied = 0
missing = 0

for _, row in sample_df.iterrows():
    img_name = row["patientId"] + ".png"
    src = os.path.join(source_folder, img_name)
    dst = os.path.join(target_folder, img_name)

    if os.path.exists(src):
        shutil.copy(src, dst)
        copied += 1
    else:
        missing += 1

sample_df.to_csv("dataset_sample/sample_labels.csv", index=False)

print("Copied images:", copied)
print("Missing images:", missing)
print("Images now in sample folder:", len(os.listdir(target_folder)))

Sample shape: (1000, 2)
Target
1    500
0    500
Name: count, dtype: int64
Copied images: 1000
Missing images: 0
Images now in sample folder: 1000


In [17]:
import os

print("Sample folder exists:", os.path.exists("dataset_sample/images"))
print("Number of copied images:", len(os.listdir("dataset_sample/images")))
print("First 5 copied images:", os.listdir("dataset_sample/images")[:5])

Sample folder exists: True
Number of copied images: 1000
First 5 copied images: ['0100515c-5204-4f31-98e0-f35e4b00004a.png', '0101174b-6643-4d4e-b4ba-b6d41d0ce46a.png', '012a5620-d082-4bb8-9b3b-e72d8938000c.png', '01a7353d-25bb-4ff8-916b-f50dd541dccf.png', '01f3abc2-33c7-4ea2-a599-dc49b76fcfae.png']


In [18]:
import cv2
import numpy as np
import pandas as pd
import os

df = pd.read_csv("dataset_sample/sample_labels.csv")

images = []
labels = []

for _, row in df.iterrows():
    img_name = row["patientId"] + ".png"
    img_path = os.path.join("dataset_sample/images", img_name)

    img = cv2.imread(img_path)
    if img is not None:
        img = cv2.resize(img, (224, 224))
        img = img / 255.0
        images.append(img)
        labels.append(row["Target"])

X = np.array(images)
y = np.array(labels)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1000, 224, 224, 3)
y shape: (1000,)


In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

Train: (800, 224, 224, 3) (800,)
Test : (200, 224, 224, 3) (200,)


In [21]:
import os

print("Sample folder exists:", os.path.exists("dataset_sample/images"))
print("Number of copied images:", len(os.listdir("dataset_sample/images")))
print("First 5 copied images:", os.listdir("dataset_sample/images")[:5])

Sample folder exists: True
Number of copied images: 1000
First 5 copied images: ['0100515c-5204-4f31-98e0-f35e4b00004a.png', '0101174b-6643-4d4e-b4ba-b6d41d0ce46a.png', '012a5620-d082-4bb8-9b3b-e72d8938000c.png', '01a7353d-25bb-4ff8-916b-f50dd541dccf.png', '01f3abc2-33c7-4ea2-a599-dc49b76fcfae.png']


In [22]:
import cv2
import numpy as np
import pandas as pd
import os

df = pd.read_csv("dataset_sample/sample_labels.csv")

images = []
labels = []

for _, row in df.iterrows():
    img_name = row["patientId"] + ".png"
    img_path = os.path.join("dataset_sample/images", img_name)

    img = cv2.imread(img_path)
    if img is not None:
        img = cv2.resize(img, (224, 224))
        img = img / 255.0
        images.append(img)
        labels.append(row["Target"])

X = np.array(images)
y = np.array(labels)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1000, 224, 224, 3)
y shape: (1000,)


In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

Train: (800, 224, 224, 3) (800,)
Test : (200, 224, 224, 3) (200,)


In [5]:
import numpy as np
import pandas as pd
import cv2
import sklearn

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("OpenCV:", cv2.__version__)
print("scikit-learn:", sklearn.__version__)

NumPy: 2.2.6
Pandas: 2.3.3
OpenCV: 4.13.0
scikit-learn: 1.7.2


In [27]:
import os
import shutil
import cv2
import numpy as np
import pandas as pd

In [28]:
X_flat = X.reshape(X.shape[0], -1)
print("Flattened shape:", X_flat.shape)

Flattened shape: (1000, 150528)


In [29]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X_flat, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy : 0.755
Precision: 0.7628865979381443
Recall   : 0.74
F1-score : 0.751269035532995

Confusion Matrix:
 [[77 23]
 [26 74]]

Classification Report:
               precision    recall  f1-score   support

           0       0.75      0.77      0.76       100
           1       0.76      0.74      0.75       100

    accuracy                           0.76       200
   macro avg       0.76      0.76      0.75       200
weighted avg       0.76      0.76      0.75       200



In [30]:
import joblib

joblib.dump(model, "pneumonia_model.pkl")
print("Model saved successfully")

Model saved successfully
